# G1 + cart on uneven terrain
Train the Unitree G1 (Genesis + rsl-rl) to push a cart at a commanded velocity over bumpy ground. Run the cells top to bottom on a **GPU runtime** (re-run the EGL cell after every reconnect).

In [ ]:
!nvidia-smi

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

In [ ]:
%pip install -q genesis-world "rsl-rl-lib>=5.0.0" mediapy tensorboard

In [ ]:
import os
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
ICD_CONFIG_CONTENT = """{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
"""
with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write(ICD_CONFIG_CONTENT)

In [ ]:
!git clone --depth 1 https://github.com/google-deepmind/mujoco_menagerie.git /content/mujoco_menagerie

import os

model_dir = "/content/mujoco_menagerie/unitree_g1"

print(os.listdir(model_dir))
print(os.path.exists(f"{model_dir}/g1.xml"))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
LOG_ROOT = '/content/drive/MyDrive/g1_logs'  # logs, checkpoints and cfgs.pkl are stored here
os.makedirs(LOG_ROOT, exist_ok=True)
print('LOG_ROOT =', LOG_ROOT)

## Training on uneven ground

In [ ]:
# from scratch (terrain amplitude --bump-height in m; upper body driven)
!python -u g1_cart_rough_train.py --log-root {LOG_ROOT} -b 4096 --max-iterations 1500 --bump-height 0.02

In [ ]:
!python -u g1_cart_rough_train.py --log-root {LOG_ROOT} --resume -b 4096 --max-iterations 1500

In [ ]:
# warm start from the flat-ground cart policy (same observations / actions), counting iterations from 0
# !python -u g1_cart_rough_train.py --log-root {LOG_ROOT} -b 4096 --max-iterations 1500 --init-from {LOG_ROOT}/g1-cart-rough/model_1400.pt

## Evaluation / video

In [ ]:
!python -u g1_cart_rough_eval.py --log-root {LOG_ROOT} --vx 0.6 --record --steps 750 --video g1_cart_rough.mp4

In [ ]:
import mediapy as media
media.show_video(media.read_video('g1_cart_rough.mp4'), fps=50)

In [ ]:
# several commands in a row (see DEMO_SCHEDULE in g1_cart_eval.py); other terrain: --bump-height 0.03 --terrain-seed 5
!python -u g1_cart_rough_eval.py --log-root {LOG_ROOT} --demo --video g1_cart_rough_demo.mp4

In [ ]:
import mediapy as media
media.show_video(media.read_video('g1_cart_rough_demo.mp4'), fps=50)

In [ ]:
!python -u g1_cart_rough_eval.py --log-root {LOG_ROOT} -e g1-cart --vx 0.4 --record --steps 750 --video g1_cart_on_rough_vx04.mp4


In [ ]:
import mediapy as media
media.show_video(media.read_video('g1_cart_on_rough_vx04.mp4'), fps=50)

## Compare models (velocity profiles)

In [ ]:
# compare models on the same terrain: cart velocity profiles (command vx = 0.2 / 0.4 / 0.6)
!python -u g1_cart_compare.py --log-root {LOG_ROOT} --models g1-cart g1-cart-rough --vx-list 0.2 0.4 0.6 --duration 12
from IPython.display import Image
Image('cart_compare.png')